In [1]:
import torch
!pip uninstall -y sympy
!pip install sympy==1.13.3
from torch import nn
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms, models
import torch.optim as optim
import requests
import random
import kagglehub
import datetime
import os
import json
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict
from timeit import default_timer as timer


Found existing installation: sympy 1.14.0
Uninstalling sympy-1.14.0:
  Successfully uninstalled sympy-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 69.6 MB/s eta 0:00:00


In [3]:
torch.manual_seed(134)
random.seed(134)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
image_path = Path(kagglehub.dataset_download("mengcius/cinic10"))
train_dir = image_path / "train"
valid_dir = image_path / "valid"
test_dir = image_path / "test"

100%|██████████| 754M/754M [00:06<00:00, 119MB/s]

Extracting files...


In [ ]:
data_transform = transforms.Compose([transforms.ToTensor()])

train_data = datasets.ImageFolder(
    root=train_dir, transform=data_transform, target_transform=None
)

valid_data = test_data = datasets.ImageFolder(root=valid_dir, transform=data_transform)

test_data = datasets.ImageFolder(root=test_dir, transform=data_transform)

In [ ]:
def build_resnet18(num_classes=10, weights="IMAGENET1K_V1"):
    model = models.resnet18(weights=weights)
    # zmiany ktore pozwalją lepiej dzialac na obrazkach 32x32, bo defaultowo sa 224x224
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()

    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

In [ ]:
def build_densenet121(num_classes=10):
    model = models.densenet121(weights=None)
    # zmiany ktore pozwalją lepiej dzialac na obrazkach 32x32, bo defaultowo sa 224x224
    # 1. Modify the first convolutional layer
    # Original: nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.features.conv0 = nn.Conv2d(
        3, 64, kernel_size=3, stride=1, padding=1, bias=False
    )

    # 2. Remove the first MaxPool (it reduces 32x32 to 16x16 too early)
    model.features.pool0 = nn.Identity()

    # 3. Modify the final fully connected layer for 10 classes
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Linear(num_ftrs, num_classes)

    return model

In [ ]:
def train_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
):
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)

    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc


def test_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
):
    model.eval()
    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            test_pred_logits = model(X)
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += (test_pred_labels == y).sum().item() / len(test_pred_labels)

    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [ ]:
def train(
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: torch.nn.Module,
    epochs: int,
    save_dir: str,
    scheduler=None,
):

    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    for epoch in tqdm(range(epochs)):
        best_acc_test = 0
        train_loss, train_acc = train_step(
            model=model,
            dataloader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
        )
        test_loss, test_acc = test_step(
            model=model, dataloader=test_dataloader, loss_fn=loss_fn
        )

        # if scheduler is not None:
        #     scheduler.step()
        #     current_lr = optimizer.param_groups[0]['lr']
        # else:
        #     current_lr = "N/A"

        if test_acc > best_acc_test:
            best_acc_test = test_acc
            best_model = model

        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"valid_loss: {test_loss:.4f} | "
            f"valid_acc: {test_acc:.4f}"
        )
        results["train_loss"].append(
            train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss
        )
        results["train_acc"].append(
            train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc
        )
        results["test_loss"].append(
            test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss
        )
        results["test_acc"].append(
            test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc
        )

    torch.save(best_model.state_dict(), save_dir)
    return results

In [ ]:
# take 10% of train/valid data

class_indices_train = defaultdict(list)
class_indices_valid = defaultdict(list)

for i, (_, label) in enumerate(train_data):
    class_indices_train[label].append(i)

for i, (_, label) in enumerate(valid_data):
    class_indices_valid[label].append(i)

selected_train = []
selected_valid = []
for label, inds in class_indices_train.items():
    random.shuffle(inds)
    selected_train.extend(inds[: int(0.1 * len(inds))])

for label, inds in class_indices_valid.items():
    random.shuffle(inds)
    selected_valid.extend(inds[: int(0.1 * len(inds))])

subset_train = Subset(train_data, selected_train)
subset_valid = Subset(valid_data, selected_valid)

train_dataloader = DataLoader(
    subset_train, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True
)
valid_dataloader = DataLoader(
    subset_valid, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True
)

In [ ]:
torch.manual_seed(134)
torch.cuda.manual_seed(134)

BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()
NUM_EPOCHS = 40
LEARNING_RATE = 0.01
model = build_densenet121(10).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

train_dataloader = DataLoader(
    dataset=train_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True
)

valid_dataloader = DataLoader(
    dataset=test_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False
)

test_dataloader = DataLoader(
    dataset=test_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False
)

now = str(datetime.datetime.now())
result_path_model = "drive/MyDrive/data/model" + now
result_path = "drive/MyDrive/data/res" + now + ".json"

start_time = timer()
model_results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=valid_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=NUM_EPOCHS,
    save_dir=result_path_model,
    scheduler=None,
)
end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

  0%|          | 0/40 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.6140 | train_acc: 0.4010 | valid_loss: 1.4869 | valid_acc: 0.4618
Epoch: 2 | train_loss: 1.3063 | train_acc: 0.5228 | valid_loss: 1.5000 | valid_acc: 0.4606
Epoch: 3 | train_loss: 1.1592 | train_acc: 0.5802 | valid_loss: 1.5584 | valid_acc: 0.4447
Epoch: 4 | train_loss: 1.0516 | train_acc: 0.6234 | valid_loss: 1.6277 | valid_acc: 0.4509
Epoch: 5 | train_loss: 0.9581 | train_acc: 0.6575 | valid_loss: 1.1599 | valid_acc: 0.5934
Epoch: 6 | train_loss: 0.8736 | train_acc: 0.6879 | valid_loss: 1.1852 | valid_acc: 0.5964
Epoch: 7 | train_loss: 0.7995 | train_acc: 0.7151 | valid_loss: 1.0656 | valid_acc: 0.6353
Epoch: 8 | train_loss: 0.7320 | train_acc: 0.7394 | valid_loss: 0.9992 | valid_acc: 0.6521
Epoch: 9 | train_loss: 0.6621 | train_acc: 0.7662 | valid_loss: 1.2519 | valid_acc: 0.5860
Epoch: 10 | train_loss: 0.5964 | train_acc: 0.7895 | valid_loss: 1.3175 | valid_acc: 0.5823
Epoch: 11 | train_loss: 0.5323 | train_acc: 0.8127 | valid_loss: 1.2295 | valid_acc: 0.62

In [ ]:
model_cur = build_densenet121().to("cuda")
state_dict = torch.load(result_path_model)
model_cur.load_state_dict(state_dict)
test_loss, test_acc = test_step(model_cur, test_dataloader, loss_fn)

result_data = {
    "results": model_results,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "time_s": end_time - start_time,
    "learnign_rate": LEARNING_RATE,
    "optimizer": "SGD",
    "data_augmentation": "None",
    "test_loss": test_loss,
    "test_acc": test_acc,
}

with open(result_path, "w") as fp:
    json.dump(result_data, fp)

In [16]:
test_loss, test_acc = test_step(model, test_dataloader, loss_fn)
print(test_loss, test_acc)

1.27080384504118 0.7991468183434056
